# SSM extension: DDM × Flexible PMC + RDM × Flexible PMC

Sequential-sampling-model versions of the paper's main behavioural
comparison (`comprehensive_model_comparison.ipynb`). The SSMs add a
WFPT/race likelihood over the joint (choice, RT) data, on top of the
same Bayesian-observer front-end used by the Flexible PMC model.
Following bauer's lesson 8: by adding the RT likelihood, the SSMs
should produce **tighter posteriors** on the shared cognitive
parameters (the noise splines, the priors) without sacrificing
choice fit.

## SSM ↔ PMC analogy

| PMC variant (paper Table 1) | DDM analogue (this notebook) | RDM analogue |
|---|---|---|
| `flexible2_null` (no TMS effect) | `ddm_flexible_null` | `rdm_flexible_null` |
| `flexible2b` (TMS on perceptual noise) | `ddm_flexible_perception` | `rdm_flexible_perception` |
| `flexible2a` (TMS on memory noise) | `ddm_flexible_memory` | `rdm_flexible_memory` |
| `flexible2` (TMS on both noise terms) | `ddm_flexible` | `rdm_flexible` |

Each row uses the **same** 5-spline noise function over magnitude,
the **same** subject-level random effects, the **same** prior beliefs
over risky / safe payoff distributions. The only structural
difference is the likelihood: Bernoulli(choice) for PMC vs.
WFPT/race(choice, RT) for SSM.

Two extra DDM/RDM variants probe an alternative mechanism — caution
shift via the accumulator threshold `a` — that the choice-only PMC
model can't distinguish from noise increase:

| Extra variant | TMS regressor |
|---|---|
| `*_threshold` | threshold `a` only (no noise effect) |
| `*_noise_threshold` | both noise splines and `a` |

In [ ]:
bids_folder = '/data/ds-tmsrisk/'

import os.path as op
import arviz as az
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Load traces

In [ ]:
# Model labels: 8 SSM (DDM + RDM × 4-variant analogue of paper Table 1)
# plus 4 extras testing the caution-shift alternative.
ddm_labels = ['ddm_flexible_null', 'ddm_flexible_perception',
              'ddm_flexible_memory', 'ddm_flexible',
              'ddm_flexible_threshold', 'ddm_flexible_noise_threshold']
rdm_labels = [l.replace('ddm_', 'rdm_') for l in ddm_labels]

# For cross-family reference: the 4 paper-canonical PMC labels.
pmc_labels = ['flexible2_null', 'flexible2b', 'flexible2a', 'flexible2']

all_labels = ddm_labels + rdm_labels + pmc_labels

# Pretty names — mirror the analogy in the markdown above
pmc_pretty = {
    'flexible2_null': 'PMC (null)',
    'flexible2b':     'PMC (TMS on perceptual)',
    'flexible2a':     'PMC (TMS on memory)',
    'flexible2':      'PMC (TMS on both)',
}
ddm_pretty = {
    'ddm_flexible_null':            'DDM (null)',
    'ddm_flexible_perception':      'DDM (TMS on perceptual)',
    'ddm_flexible_memory':          'DDM (TMS on memory)',
    'ddm_flexible':                 'DDM (TMS on both)',
    'ddm_flexible_threshold':       'DDM (TMS on threshold)',
    'ddm_flexible_noise_threshold': 'DDM (TMS on both + threshold)',
}
rdm_pretty = {k.replace('ddm', 'rdm'): v.replace('DDM', 'RDM')
              for k, v in ddm_pretty.items()}
mapping = {**pmc_pretty, **ddm_pretty, **rdm_pretty}

In [ ]:
idatas = {}
for label in all_labels:
    path = op.join(bids_folder, 'derivatives', 'cogmodels', f'model-{label}_trace.netcdf')
    if not op.exists(path):
        print(f'  MISSING: {label}')
        continue
    idatas[label] = az.from_netcdf(path)

print(f'Loaded {len(idatas)} / {len(all_labels)} traces')
labeled = {mapping[k]: v for k, v in idatas.items()}

## ELPD comparison (Table-1 analogue)

Same structure as `comprehensive_model_comparison.ipynb`: pool every
model into one `az.compare(..., ic='loo')` call. The within-family
rankings below partition this for legibility.

In [ ]:
comparison = az.compare(labeled, ic='loo')
comparison

In [ ]:
az.plot_compare(comparison, figsize=(8, 6))
plt.tight_layout()

In [ ]:
def family(labels):
    return {mapping[k]: idatas[k] for k in labels if k in idatas}

for name, family_labels in [('Paper PMC family',  pmc_labels),
                             ('DDM family',         ddm_labels),
                             ('RDM family',         rdm_labels)]:
    fam = family(family_labels)
    if len(fam) < 2:
        print(f'--- {name}: skipped (only {len(fam)} traces) ---')
        continue
    print(f'\n--- {name} ---')
    display(az.compare(fam, ic='loo'))

## Posterior tightening: HDI widths vs PMC (lesson-8 style)

Per bauer's tutorial lesson 8: if the SSM is well-specified, joint
(choice, RT) likelihood should produce **tighter** posteriors on the
shared cognitive parameters than the choice-only PMC.

Compare the 'TMS on both' variant of each family: PMC `flexible2` vs
DDM `ddm_flexible` vs RDM `rdm_flexible`.

In [ ]:
def hdi_widths(idata, prefix):
    """94% HDI widths for every group-mu variable whose name starts with `prefix`."""
    rows = []
    post = idata.posterior
    for var in post.data_vars:
        if not var.startswith(prefix) or not var.endswith('_mu'):
            continue
        hdi = az.hdi(post[var], hdi_prob=0.94)[var].values
        flat = hdi.reshape(-1, 2)
        widths = flat[:, 1] - flat[:, 0]
        for i, w in enumerate(widths):
            rows.append({'parameter': var, 'index': i, 'width': float(w)})
    return pd.DataFrame(rows)

# Look at the noise splines specifically — they're the shared front-end.
pmc_w = hdi_widths(idatas['flexible2'], 'memory_noise_sd_spline').assign(model='PMC')
frames = [pmc_w]
for k, name in [('ddm_flexible', 'DDM'), ('rdm_flexible', 'RDM')]:
    if k in idatas:
        frames.append(hdi_widths(idatas[k], 'memory_noise_sd_spline').assign(model=name))
widths = pd.concat(frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(6, 4))
sns.stripplot(data=widths, x='parameter', y='width', hue='model', dodge=True, ax=ax, alpha=0.6)
ax.set_title('Group-μ 94% HDI width — memory noise splines')
ax.set_ylabel('HDI width (lower = tighter)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

## What to look for

1. **Within each family, the bare (`*_flexible`) variant is close to or beats `*_null`.** Confirms the noise effect is detectable jointly with RT data, not just from choices.
2. **`*_threshold` ranks below `*_perception` / `*_memory` / bare.** Rules out the simpler caution-shift account: cTBS doesn't just shift accumulator boundaries, it inflates perceptual noise.
3. **`*_noise_threshold` doesn't decisively beat `*_flexible`.** Tells us we don't need to add a caution-shift component to the paper's claim.
4. **The HDI-width plot shows DDM/RDM bars below the PMC bar.** That's the lesson-8 promise: RT information tightens the noise-spline posteriors.